# Release16 Guarded Reward Short PPO Screen

This notebook executes the next action from `release16_guarded_reward_checkpoint_objective_v1`: a short fold-local BC/PPO training screen using the validated guarded reward patch.

- Canonical baseline: `soxlbacktest_policy_reimplementation_v1`
- Direct comparison baseline: `release16_bull_reentry_participation_objective_replay_v1::bull65_hold3`
- Role: `diagnose`; no actor promotion or retention is allowed in this notebook.
- Target window: `wf_2020`, `wf_2022`, `wf_2023_2024`
- Primary gate: relative return >= 30pp and MaxDD <= 20% on every evaluated fold
- Strict context: MaxDD <= 15% is retained as risk context but is not the primary gate
- Changed layer: `reward`, `bc_to_ppo_handoff`, `model_selection`
- External reference: converted SOXLBacktest qlib baseline plus release16 official replay handoff

Historical success case: the prior bull-reentry replay reduced official gate loss and made `wf_2022` pass the primary gate.

Historical failure case: prior reward PPO screens often improved return but expanded MaxDD, especially in stressed folds, so this notebook has a PPO-vs-BC drift guard.

Why this is not a duplicate: it applies the newly screened guarded drawdown-drift/trend-preservation patch to the latest release16 bull65 fold-local target contract and compares BC initialization against reward PPO on the official daily path metric basis.


In [1]:
from __future__ import annotations

import copy
import json
import math
import random
import sys
from dataclasses import asdict, replace
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

RUN_CWD = Path.cwd().resolve()
for candidate_root in (RUN_CWD, *RUN_CWD.parents):
    if (candidate_root / "SPEC.md").exists() and (candidate_root / "src" / "soxl_quant").exists():
        ROOT = candidate_root
        break
else:
    raise RuntimeError(f"could not locate repo root from {RUN_CWD}")
for module_path in (ROOT, ROOT / "src"):
    module_path_text = str(module_path)
    if module_path_text not in sys.path:
        sys.path.insert(0, module_path_text)

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

from src.soxl_quant.app.checkpoint_daily_export import (
    build_checkpoint_candidate_daily_export,
    metrics_from_official_backtest,
)
from src.soxl_quant.app.constrained_checkpoint_selector import (
    ConstrainedCheckpointSelectorConfig,
    score_constrained_checkpoints,
    summarize_constrained_checkpoint_selection,
)
from src.soxl_quant.app.selection import train_with_selection
from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCSampleWeightConfig,
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
    summarize_bc_slice_fit,
)
from src.soxl_quant.backtest import backtest_profile
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import RewardConfig, load_profile
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, init_qlib, segment_frame
from src.soxl_quant.policy.environment import build_initial_states
from src.soxl_quant.policy.evaluation import evaluate_split
from src.soxl_quant.policy.observation import observation_dim_for_profile
from src.soxl_quant.policy.policy_builder import build_policy
from src.soxl_quant.policy.positioning import action_levels_for_profile
from src.soxl_quant.settings import QLIB_DATA_PATH

patch_gym_compat()
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
EXPERIMENT = "release16_guarded_reward_short_ppo_screen_v1"
CARD = ROOT / "runtime" / "reward" / "card" / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
DAILY_DIR = CARD / "official_daily_paths"
DAILY_DIR.mkdir(parents=True, exist_ok=True)

PROFILE_NAME = "sparse_release_bc_ppo_screen_v1"
SEED = 20260615
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 20.0
STRICT_MAX_DD = 15.0
MAX_DD_DRIFT_ALLOWANCE = 2.0
RELATIVE_DRIFT_ALLOWANCE = -5.0
BULL_SLICE_MIN_SAMPLES = 5
BULL_SLICE_MIN_ACCURACY = 0.60

SOURCE_REWARD_CARD = ROOT / "runtime" / "reward" / "card" / "release16_guarded_reward_checkpoint_objective_v1"
SOURCE_BULL_CARD = ROOT / "runtime" / "model_selection" / "card" / "release16_bull_reentry_participation_objective_replay_v1"
TARGET_CONTRACT_PATH = SOURCE_BULL_CARD / "bull65_hold3_target_contract.csv"
PRIOR_BULL_METRICS_PATH = SOURCE_BULL_CARD / "bc_official_replay_metrics.csv"
RETENTION_GATE_PATH = ROOT / "runtime" / "model_selection" / "card" / "model_retention_walkforward_gate_v1" / "summary.json"

TRAIN_CONFIG = StateObjectiveBCTrainingConfig(steps=400, lr=0.001, batch_size=128, seed=SEED)
SELECTED_WEIGHT_CONFIG = StateObjectiveBCSampleWeightConfig(
    rebound_multiplier=6.0,
    downtrend_multiplier=2.0,
    release_gate_multiplier=64.0,
    objective_weight_power=0.0,
    class_balanced_beta=0.99,
    weight_cap=80.0,
)

WALK_FORWARD_WINDOWS = {
    "wf_2020": {
        "target_fold": "wf_2020_train",
        "train": ("2010-03-11", "2018-12-31"),
        "valid": ("2019-01-01", "2019-12-31"),
        "test": ("2020-01-01", "2020-12-31"),
    },
    "wf_2022": {
        "target_fold": "wf_2022_train",
        "train": ("2010-03-11", "2020-12-31"),
        "valid": ("2021-01-01", "2021-12-31"),
        "test": ("2022-01-01", "2022-12-31"),
    },
    "wf_2023_2024": {
        "target_fold": "wf_2023_2024_train",
        "train": ("2010-03-11", "2022-12-31"),
        "valid": ("2023-01-01", "2023-12-31"),
        "test": ("2024-01-01", "2024-12-31"),
    },
}


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        try:
            return str(value.relative_to(ROOT))
        except ValueError:
            return str(value)
    if pd.isna(value):
        return None
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + chr(10), encoding="utf-8")


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}


def rel(path: Path) -> str:
    return str(path.relative_to(ROOT))


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def normalize_bool_series(series: pd.Series) -> pd.Series:
    if series.dtype == object:
        return series.fillna(False).astype(str).str.lower().isin({"true", "1", "yes", "y", "t"})
    return series.fillna(False).astype(bool)

missing_inputs = [str(p) for p in (SOURCE_REWARD_CARD, SOURCE_BULL_CARD, TARGET_CONTRACT_PATH, PRIOR_BULL_METRICS_PATH) if not p.exists()]
if missing_inputs:
    raise FileNotFoundError(missing_inputs)
print("root", ROOT)
print("card", CARD)


root /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix
card /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/reward/card/release16_guarded_reward_short_ppo_screen_v1


In [3]:
base_profile = load_profile(PROFILE_NAME)
reward_source_summary = load_json(SOURCE_REWARD_CARD / "summary.json")
reward_source_selection = load_json(SOURCE_REWARD_CARD / "selection.json")
reward_patch = dict(reward_source_summary.get("reward_objective_patch", {}))
missing_patch_keys = sorted(set(reward_patch) - set(RewardConfig.__dataclass_fields__))
if missing_patch_keys:
    raise ValueError(f"reward patch keys missing from RewardConfig: {missing_patch_keys}")
profile = replace(base_profile, reward=replace(base_profile.reward, **reward_patch))

prior_bull_summary = load_json(SOURCE_BULL_CARD / "summary.json")
prior_bull_selection = load_json(SOURCE_BULL_CARD / "selection.json")
prior_bull_metrics = pd.read_csv(PRIOR_BULL_METRICS_PATH)
targets = pd.read_csv(TARGET_CONTRACT_PATH)
targets["datetime"] = pd.to_datetime(targets["datetime"]).dt.normalize()
for col in (
    "training_label_allowed",
    "is_rebound_cluster",
    "is_downtrend_state",
    "release_gate_state",
    "drawdown_control_state",
    "bull_reentry_target_state",
):
    if col in targets.columns:
        targets[col] = normalize_bool_series(targets[col])

retention_summary = load_json(RETENTION_GATE_PATH).get("summary", {})
print("profile", profile.name)
print("reward_patch", reward_patch)
print("source_reward_decision", reward_source_summary.get("decision"))
print("prior_bull_decision", prior_bull_summary.get("decision"))


profile sparse_release_bc_ppo_screen_v1
reward_patch {'drawdown_drift_penalty': 1.5, 'drawdown_drift_position_threshold': 0.15, 'drawdown_drift_reward_enabled': True, 'drawdown_drift_threshold_pct': 18.0, 'trend_preservation_floor_enabled': True, 'trend_preservation_floor_penalty': 1.0, 'trend_preservation_floor_position': 0.25, 'trend_preservation_lookback': 21, 'trend_preservation_min_buyhold_return_pct': 5.0}
source_reward_decision guarded_reward_recipe_ready_for_short_training_no_promotion
prior_bull_decision bull_reentry_participation_objective_partial_improvement_no_promotion


In [4]:
coverage_rows: list[dict[str, Any]] = []
for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    train_start = pd.Timestamp(spec["train"][0])
    train_end = pd.Timestamp(spec["train"][1])
    valid_start = pd.Timestamp(spec["valid"][0])
    test_start = pd.Timestamp(spec["test"][0])
    all_fold_targets = targets[targets["fold"].astype(str).eq(spec["target_fold"])].copy()
    train_targets = all_fold_targets[
        all_fold_targets["datetime"].between(train_start, train_end)
        & all_fold_targets["training_label_allowed"].fillna(False).astype(bool)
    ].copy()
    valid_overlap = int((all_fold_targets["datetime"].between(valid_start, pd.Timestamp(spec["valid"][1]))).sum())
    test_overlap = int((all_fold_targets["datetime"] >= test_start).sum())
    bull_state = train_targets.get("bull_reentry_target_state", pd.Series(False, index=train_targets.index)).fillna(False).astype(bool)
    coverage_rows.append(
        {
            "fold": fold_name,
            "target_fold": spec["target_fold"],
            "target_rows_available": int(len(all_fold_targets)),
            "train_target_rows": int(len(train_targets)),
            "bull_reentry_target_rows": int(bull_state.sum()),
            "train_start": spec["train"][0],
            "train_end": spec["train"][1],
            "valid_start": spec["valid"][0],
            "valid_end": spec["valid"][1],
            "test_start": spec["test"][0],
            "test_end": spec["test"][1],
            "valid_overlap_rows_in_contract": valid_overlap,
            "test_overlap_rows_in_contract": test_overlap,
            "train_target_start": str(train_targets["datetime"].min().date()) if not train_targets.empty else "",
            "train_target_end": str(train_targets["datetime"].max().date()) if not train_targets.empty else "",
            "skip": bool(train_targets.empty or test_overlap > 0),
            "skip_reason": "" if (not train_targets.empty and test_overlap == 0) else "missing_train_targets_or_test_overlap",
        }
    )
coverage_frame = pd.DataFrame(coverage_rows)
coverage_frame.to_csv(CARD / "coverage.csv", index=False)
coverage = {
    "experiment": EXPERIMENT,
    "profile": PROFILE_NAME,
    "source_reward_card": rel(SOURCE_REWARD_CARD),
    "source_bull_card": rel(SOURCE_BULL_CARD),
    "target_contract": rel(TARGET_CONTRACT_PATH),
    "reward_patch": reward_patch,
    "reward_source_decision": reward_source_summary.get("decision"),
    "reward_training_allowed_next": bool(reward_source_summary.get("short_training_allowed_next", False)),
    "prior_bull_decision": prior_bull_summary.get("decision"),
    "windows": coverage_rows,
    "sample_weight_config": asdict(SELECTED_WEIGHT_CONFIG),
    "train_config": asdict(TRAIN_CONFIG),
}
write_json(CARD / "coverage.json", coverage)
display(coverage_frame)


,fold,target_fold,target_rows_available,train_target_rows,bull_reentry_target_rows,train_start,train_end,valid_start,valid_end,test_start,test_end,valid_overlap_rows_in_contract,test_overlap_rows_in_contract,train_target_start,train_target_end,skip,skip_reason
0,wf_2020,wf_2020_train,2470,2218,14,2010-03-11,2018-12-31,2019-01-01,2019-12-31,2020-01-01,2020-12-31,252,0,2010-03-11,2018-12-31,False,
1,wf_2022,wf_2022_train,505,253,1,2010-03-11,2020-12-31,2021-01-01,2021-12-31,2022-01-01,2022-12-31,252,0,2020-01-02,2020-12-31,False,
2,wf_2023_2024,wf_2023_2024_train,756,756,0,2010-03-11,2022-12-31,2023-01-01,2023-12-31,2024-01-01,2024-12-31,0,0,2020-01-02,2022-12-30,False,


In [5]:
def evaluate_actor_path(
    actor,
    eval_df: pd.DataFrame,
    price_series: pd.Series,
    profile,
    test_seg: tuple[str, str],
    *,
    fold: str,
    candidate: str,
    candidate_kind: str,
) -> dict[str, Any]:
    feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)
    split_metrics = evaluate_split(
        actor,
        eval_df,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        execution_config=profile.execution,
        asset_idx=0,
        action_levels=action_levels_for_profile(profile),
        observation_window=profile.model.observation_window,
        hard_floor=profile.model.hard_floor,
        use_hybrid_overlay=profile.model.use_hybrid_overlay,
        delta_levels=np.asarray(profile.model.delta_levels, dtype=np.float32) if profile.model.use_hybrid_overlay else None,
        feature_columns=feature_columns,
        mode_router_config=profile.mode_router,
        position_mapping=profile.model.position_mapping,
    )
    bt = backtest_profile(
        actor,
        eval_df,
        price_series,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        execution_config=profile.execution,
        asset_idx=0,
        feature_columns=feature_columns,
        model_config=profile.model,
        mode_router_config=profile.mode_router,
        test_seg=test_seg,
    )
    out = metrics_from_official_backtest(split_metrics=dict(split_metrics), backtest_result=bt)
    export = build_checkpoint_candidate_daily_export(
        backtest_result=bt,
        price_series=price_series,
        output_dir=DAILY_DIR,
        fold=fold,
        candidate=candidate,
        candidate_kind=candidate_kind,
        split_metrics=dict(split_metrics),
        root=ROOT,
    )
    for source_key, target_key in {
        "test_return": "return",
        "test_max_dd": "max_dd",
        "test_avg_pos": "avg_pos",
        "test_buy_and_hold_return": "buy_and_hold_return",
        "test_relative_return": "relative_return",
    }.items():
        if source_key in export.row:
            out[target_key] = float(export.row[source_key])
    out.update({k: v for k, v in export.row.items() if k not in out})
    return out


def prefixed_metrics(prefix: str, metrics: dict[str, Any]) -> dict[str, Any]:
    keep = [
        "return",
        "sharpe",
        "max_dd",
        "avg_pos",
        "stress_avg_pos",
        "buy_and_hold_return",
        "relative_return",
        "daily_path",
        "daily_path_rows",
        "metric_basis",
        "raw_backtest_return",
        "raw_backtest_max_dd",
        "metric_return_abs_diff",
        "metric_max_dd_abs_diff",
    ]
    return {f"{prefix}_{key}": metrics[key] for key in keep if key in metrics}


def gate_pass(metrics: dict[str, Any], max_dd: float = TARGET_MAX_DD) -> bool:
    return bool(float(metrics.get("relative_return", -1e9)) >= TARGET_RELATIVE_RETURN and float(metrics.get("max_dd", 1e9)) <= max_dd)


In [6]:
init_qlib(QLIB_DATA_PATH)
obs_dim = observation_dim_for_profile(profile)
fold_rows: list[dict[str, Any]] = []
history_rows: list[dict[str, Any]] = []
bc_slice_frames: list[pd.DataFrame] = []

for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    print(f"\n[fold:{fold_name}] train={spec['train']} valid={spec['valid']} test={spec['test']}")
    coverage_row = coverage_frame[coverage_frame["fold"].eq(fold_name)].iloc[0].to_dict()
    if bool(coverage_row.get("skip", False)):
        fold_rows.append({"fold": fold_name, "status": "skipped", "skip_reason": coverage_row.get("skip_reason", "")})
        continue

    dataset = build_dataset(
        train_seg=spec["train"],
        valid_seg=spec["valid"],
        test_seg=spec["test"],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    train_df = segment_frame(dataset, "train")
    valid_df = segment_frame(dataset, "valid")
    test_df = segment_frame(dataset, "test")
    price_frame = dataset.handler.price_frame.droplevel("instrument")["price"].sort_index()
    valid_price = price_frame.loc[spec["valid"][0] : spec["valid"][1]]
    test_price = price_frame.loc[spec["test"][0] : spec["test"][1]]

    train_start = pd.Timestamp(spec["train"][0])
    train_end = pd.Timestamp(spec["train"][1])
    fold_targets = targets[targets["fold"].astype(str).eq(spec["target_fold"])].copy()
    fold_targets = fold_targets[fold_targets["datetime"].between(train_start, train_end)].copy()
    fold_targets = fold_targets[fold_targets["training_label_allowed"].fillna(False).astype(bool)].copy()
    if fold_targets.empty:
        fold_rows.append({"fold": fold_name, "status": "skipped_no_targets"})
        continue

    set_seed(SEED)
    bc_result = fit_state_objective_bc_actor_from_frame(
        train_df,
        targets=fold_targets,
        profile=profile,
        train_config=TRAIN_CONFIG,
        sample_weight_config=SELECTED_WEIGHT_CONFIG,
        sample_size=0,
        rng_seed=SEED,
    )
    bc_dataset = build_state_objective_bc_dataset_from_frame(
        train_df,
        targets=fold_targets,
        profile=profile,
        sample_size=0,
    )
    bc_slice_metrics = evaluate_bc_actor_slice_fit(
        bc_result.actor,
        bc_result.obs,
        bc_result.actions,
        bc_dataset.metadata,
        slice_cols=(
            "bull_reentry_target_state",
            "release_gate_state",
            "is_rebound_cluster",
            "is_downtrend_state",
            "drawdown_control_state",
        ),
    )
    bc_slice_summary = summarize_bc_slice_fit(
        bc_slice_metrics,
        required_slices=(),
        min_accuracy=BULL_SLICE_MIN_ACCURACY,
        min_samples=BULL_SLICE_MIN_SAMPLES,
    )
    tagged_slices = bc_slice_metrics.copy()
    tagged_slices.insert(0, "fold", fold_name)
    bc_slice_frames.append(tagged_slices)

    train_pickle = CARD / f"{fold_name}_train_frame.pkl"
    train_df.to_pickle(train_pickle)
    initial_states = build_initial_states([(train_pickle, train_df, 0)], profile, smoke=False)
    if not initial_states:
        fold_rows.append({"fold": fold_name, "status": "skipped_no_initial_states"})
        continue

    policy, actor = build_policy(obs_dim, profile)
    actor.load_state_dict(copy.deepcopy(bc_result.actor.state_dict()))
    actor.eval()
    bc_test_metrics = evaluate_actor_path(
        actor,
        test_df,
        test_price,
        profile,
        spec["test"],
        fold=fold_name,
        candidate="bc_initialization",
        candidate_kind="bc_initialization",
    )
    bc_valid_metrics = evaluate_actor_path(
        actor,
        valid_df,
        valid_price,
        profile,
        spec["valid"],
        fold=f"{fold_name}_valid",
        candidate="bc_initialization",
        candidate_kind="bc_initialization",
    )

    actor.train()
    selection = train_with_selection(
        policy=policy,
        actor=actor,
        initial_states=initial_states,
        obs_dim=obs_dim,
        profile=profile,
        valid_df=valid_df,
        smoke=False,
        test_df=None,
    )
    actor.eval()
    ppo_test_metrics = evaluate_actor_path(
        actor,
        test_df,
        test_price,
        profile,
        spec["test"],
        fold=fold_name,
        candidate="reward_ppo_selected",
        candidate_kind="reward_ppo_selected",
    )
    ppo_valid_metrics = dict(selection.get("best_metrics", {}))
    for item in selection.get("history", []):
        row = dict(item)
        row["fold"] = fold_name
        history_rows.append(row)

    rel_delta = float(ppo_test_metrics.get("relative_return", 0.0) - bc_test_metrics.get("relative_return", 0.0))
    dd_delta = float(ppo_test_metrics.get("max_dd", 0.0) - bc_test_metrics.get("max_dd", 0.0))
    selected_gate = gate_pass(ppo_test_metrics, TARGET_MAX_DD)
    selected_strict_gate = gate_pass(ppo_test_metrics, STRICT_MAX_DD)
    bc_gate = gate_pass(bc_test_metrics, TARGET_MAX_DD)
    bc_strict_gate = gate_pass(bc_test_metrics, STRICT_MAX_DD)
    drift_gate = bool(rel_delta >= RELATIVE_DRIFT_ALLOWANCE and dd_delta <= MAX_DD_DRIFT_ALLOWANCE)

    bull_slice = bc_slice_metrics[bc_slice_metrics["slice"].astype(str).eq("bull_reentry_target_state")]
    bull_slice_row = bull_slice.iloc[0].to_dict() if not bull_slice.empty else {"accuracy": np.nan, "samples": 0}
    fold_rows.append(
        {
            "fold": fold_name,
            "status": "evaluated",
            "train_rows": int(len(train_df)),
            "valid_rows": int(len(valid_df)),
            "test_rows": int(len(test_df)),
            "initial_states": int(len(initial_states)),
            "target_rows": int(len(fold_targets)),
            "bull_reentry_target_rows": int(fold_targets.get("bull_reentry_target_state", pd.Series(False, index=fold_targets.index)).fillna(False).astype(bool).sum()),
            "bc_after_accuracy": float(bc_result.metrics["after_accuracy"]),
            "bc_after_loss": float(bc_result.metrics["after_loss"]),
            "bc_bull_reentry_accuracy": float(bull_slice_row.get("accuracy", np.nan)) if pd.notna(bull_slice_row.get("accuracy", np.nan)) else np.nan,
            "bc_bull_reentry_samples": int(bull_slice_row.get("samples", 0) or 0),
            "bc_slice_gate": bool(bc_slice_summary["slice_fit_gate"]),
            "ppo_valid_score": float(selection.get("best_score", 0.0)),
            "ppo_valid_return": float(ppo_valid_metrics.get("return", 0.0)),
            "ppo_valid_sharpe": float(ppo_valid_metrics.get("sharpe", 0.0)),
            "ppo_valid_max_dd": float(ppo_valid_metrics.get("max_dd", 0.0)),
            "ppo_vs_bc_relative_delta": rel_delta,
            "ppo_vs_bc_max_dd_delta": dd_delta,
            "bc_gate_20dd": bc_gate,
            "bc_strict_gate_15dd": bc_strict_gate,
            "selected_gate_20dd": selected_gate,
            "selected_strict_gate_15dd": selected_strict_gate,
            "drift_gate": drift_gate,
            "fold_gate": bool(selected_gate and drift_gate),
            **prefixed_metrics("bc_test", bc_test_metrics),
            **prefixed_metrics("selected_test", ppo_test_metrics),
            **prefixed_metrics("bc_valid", bc_valid_metrics),
        }
    )

fold_results = pd.DataFrame(fold_rows)
history = pd.DataFrame(history_rows)
bc_slice_metrics_all = pd.concat(bc_slice_frames, ignore_index=True) if bc_slice_frames else pd.DataFrame()
fold_results.to_csv(CARD / "reward_ppo_fold_results.csv", index=False)
history.to_csv(CARD / "reward_ppo_selection_history.csv", index=False)
bc_slice_metrics_all.to_csv(CARD / "bc_slice_metrics.csv", index=False)
for train_pickle in CARD.glob("wf_*_train_frame.pkl"):
    train_pickle.unlink(missing_ok=True)

display(fold_results)
display(history)


[19022:MainThread](2026-06-15 05:02:44,410) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[19022:MainThread](2026-06-15 05:02:44,816) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[19022:MainThread](2026-06-15 05:02:44,816) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/data/qlib/soxl')}



[fold:wf_2020] train=('2010-03-11', '2018-12-31') valid=('2019-01-01', '2019-12-31') test=('2020-01-01', '2020-12-31')


[19022:MainThread](2026-06-15 05:02:45,768) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


...

[19022:MainThread](2026-06-15 05:05:11,158) WARNING - qlib.online operator - [exchange.py:228] - trade unit 1 is not supported in adjusted_price mode.


backtest loop:   0%|          | 0/251 [00:00<?, ?it/s]

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rzi

,fold,status,train_rows,valid_rows,test_rows,initial_states,target_rows,bull_reentry_target_rows,bc_after_accuracy,bc_after_loss,bc_bull_reentry_accuracy,bc_bull_reentry_samples,bc_slice_gate,ppo_valid_score,ppo_valid_return,ppo_valid_sharpe,ppo_valid_max_dd,ppo_vs_bc_relative_delta,ppo_vs_bc_max_dd_delta,bc_gate_20dd,bc_strict_gate_15dd,selected_gate_20dd,selected_strict_gate_15dd,drift_gate,fold_gate,bc_test_return,bc_test_sharpe,bc_test_max_dd,bc_test_avg_pos,bc_test_stress_avg_pos,bc_test_buy_and_hold_return,bc_test_relative_return,bc_test_daily_path,bc_test_daily_path_rows,bc_test_metric_basis,bc_test_raw_backtest_return,bc_test_raw_backtest_max_dd,bc_test_metric_return_abs_diff,bc_test_metric_max_dd_abs_diff,selected_test_return,selected_test_sharpe,selected_test_max_dd,selected_test_avg_pos,selected_test_stress_avg_pos,selected_test_buy_and_hold_return,selected_test_relative_return,selected_test_daily_path,selected_test_daily_path_rows,selected_test_metric_basis,selected_test_raw_backtest_return,selected_test_raw_backtest_max_dd,selected_test_metric_return_abs_diff,selected_test_metric_max_dd_abs_diff,bc_valid_return,bc_valid_sharpe,bc_valid_max_dd,bc_valid_avg_pos,bc_valid_stress_avg_pos,bc_valid_buy_and_hold_return,bc_valid_relative_return,bc_valid_daily_path,bc_valid_daily_path_rows,bc_valid_metric_basis,bc_valid_raw_backtest_return,bc_valid_raw_backtest_max_dd,bc_valid_metric_return_abs_diff,bc_valid_metric_max_dd_abs_diff
0,wf_2020,evaluated,1965,252,253,15,2218,14,0.657038,1.243742,1.0,11,True,49.521613,27.576675,1.201092,17.307581,-37.542462,0.006615,False,False,False,False,False,False,80.375210,1.868960,21.106156,0.425202,0.200000,58.093571,22.281639,runtime/reward/card/release16_guarded_reward_s...,252,official_daily_path_rebased_eval_window,80.375210,21.106156,0.0,7.105427e-15,42.832748,1.184248,21.112771,0.398541,0.196622,58.093571,-15.260823,runtime/reward/card/release16_guarded_reward_s...,252,official_daily_path_rebased_eval_window,42.832748,21.112771,0.0,0.000000e+00,18.444818,0.690370,24.734108,0.391899,0.292308,219.174548,-200.729730,runtime/reward/card/release16_guarded_reward_s...,251,official_daily_path_rebased_eval_window,18.444818,24.734108,0.0,3.552714e-15
1,wf_2022,evaluated,2470,252,251,21,253,1,0.789683,1.115628,1.0,1,True,30.082365,12.365753,0.949489,10.609808,0.000000,0.000000,True,False,True,False,True,True,-14.137212,-0.978142,15.072639,0.087837,0.080378,-86.588072,72.450860,runtime/reward/card/release16_guarded_reward_s...,250,official_daily_path_rebased_eval_window,-14.137212,15.072639,0.0,5.329071e-15,-14.137212,-0.978142,15.072639,0.087837,0.080378,-86.588072,72.450860,runtime/reward/card/release16_guarded_reward_s...,250,official_daily_path_rebased_eval_window,-14.137212,15.072639,0.0,5.329071e-15,4.980515,0.388885,15.250157,0.202323,0.066667,122.893669,-117.913154,runtime/reward/card/release16_guarded_reward_s...,251,official_daily_path_rebased_eval_window,4.980515,15.250157,0.0,3.552714e-15
2,wf_2023_2024,evaluated,2973,250,252,23,756,0,0.715232,1.187326,NaN,0,True,64.979156,36.598468,1.524014,17.496515,0.000000,0.000000,False,False,False,False,True,False,13.945428,0.799003,13.959434,0.197766,0.097359,-0.356635,14.302062,runtime/reward/card/release16_guarded_reward_s...,251,official_daily_path_rebased_eval_window,13.945428,13.959434,0.0,3.552714e-15,13.945428,0.799003,13.959434,0.197766,0.097359,-0.356635,14.302062,runtime/reward/card/release16_guarded_reward_s...,251,official_daily_path_rebased_eval_window,13.945428,13.959434,0.0,3.552714e-15,36.574209,1.201045,23.695008,0.404455,0.000000,244.230795,-207.656586,runtime/reward/card/release16_guarded_reward_s...,249,official_daily_path_rebased_eval_window,36.574209,23.695008,0.0,3.552714e-15


,iter,score,return,sharpe,max_dd,avg_pos,stress_avg_pos,stress_days,buy_and_hold_return,relative_return,fold
0,0,38.953006,21.182395,0.992376,17.307581,0.321859,0.292308,13.0,98.480336,-77.297941,wf_2020
1,4,49.521613,27.576675,1.201092,17.307581,0.345157,0.273077,13.0,98.480336,-70.903661,wf_2020
2,0,30.082365,12.365753,0.949489,10.609808,0.213416,0.066667,3.0,78.905912,-66.540159,wf_2022
3,4,26.042733,8.514829,0.919623,7.204625,0.149149,0.066667,3.0,78.905912,-70.391083,wf_2022
4,0,64.979156,36.598468,1.524014,17.496515,0.332672,0.000000,0.0,79.498558,-42.900090,wf_2023_2024
5,4,61.761932,32.935861,1.535451,15.691269,0.274339,0.000000,0.0,79.498558,-46.562697,wf_2023_2024


In [7]:
evaluated = fold_results[fold_results["status"].eq("evaluated")].copy() if not fold_results.empty else pd.DataFrame()
constraint_rows: list[dict[str, Any]] = []
if not evaluated.empty:
    for row in evaluated.to_dict(orient="records"):
        for prefix, candidate in (("bc_test", "bc_initialization"), ("selected_test", "reward_ppo_selected")):
            constraint_rows.append(
                {
                    "candidate": candidate,
                    "window": row["fold"],
                    "relative_return": row.get(f"{prefix}_relative_return", float("nan")),
                    "max_dd": row.get(f"{prefix}_max_dd", float("nan")),
                }
            )
constraint_metrics = pd.DataFrame(constraint_rows)
constraint_scores = score_constrained_checkpoints(
    constraint_metrics,
    reference_candidate="bc_initialization",
    config=ConstrainedCheckpointSelectorConfig(
        target_relative_return_pp=TARGET_RELATIVE_RETURN,
        target_max_dd_pct=TARGET_MAX_DD,
        relative_gap_weight=1.0,
        max_dd_gap_weight=10.0,
        failed_window_weight=100.0,
        relative_drift_weight=0.25,
        max_dd_drift_weight=2.0,
    ),
) if not constraint_metrics.empty else pd.DataFrame()
constraint_summary = summarize_constrained_checkpoint_selection(constraint_scores)
constraint_metrics.to_csv(CARD / "constrained_checkpoint_metrics.csv", index=False)
constraint_scores.to_csv(CARD / "constrained_checkpoint_scores.csv", index=False)

all_folds_evaluated = bool(len(evaluated) == len(WALK_FORWARD_WINDOWS))
all_selected_gate = bool(all_folds_evaluated and evaluated["selected_gate_20dd"].fillna(False).astype(bool).all())
all_drift_gate = bool(all_folds_evaluated and evaluated["drift_gate"].fillna(False).astype(bool).all())
all_fold_gate = bool(all_folds_evaluated and evaluated["fold_gate"].fillna(False).astype(bool).all())
all_bc_gate = bool(all_folds_evaluated and evaluated["bc_gate_20dd"].fillna(False).astype(bool).all())
any_selected_gate = bool((not evaluated.empty) and evaluated["selected_gate_20dd"].fillna(False).astype(bool).any())
any_drift_fail = bool((not evaluated.empty) and (~evaluated["drift_gate"].fillna(False).astype(bool)).any())
selected_gate_count = int(evaluated["selected_gate_20dd"].fillna(False).astype(bool).sum()) if not evaluated.empty else 0
bc_gate_count = int(evaluated["bc_gate_20dd"].fillna(False).astype(bool).sum()) if not evaluated.empty else 0
selected_strict_gate_count = int(evaluated["selected_strict_gate_15dd"].fillna(False).astype(bool).sum()) if not evaluated.empty else 0
bc_strict_gate_count = int(evaluated["bc_strict_gate_15dd"].fillna(False).astype(bool).sum()) if not evaluated.empty else 0

if all_fold_gate:
    decision = "release16_guarded_reward_short_ppo_screen_passed_no_promotion"
    next_mechanism = "run_full_fixed_split_walkforward_dashboard_retention_review_before_actor_promotion"
elif all_bc_gate and not all_selected_gate:
    decision = "release16_guarded_reward_short_ppo_bc_initialization_better_no_promotion"
    next_mechanism = "official_replay_bc_initialization_or_revise_reward_before_longer_ppo"
elif any_drift_fail:
    decision = "release16_guarded_reward_short_ppo_drift_failed_no_promotion"
    next_mechanism = "do_not_launch_longer_ppo_from_this_reward_patch; inspect_failed_fold_action_distribution"
elif any_selected_gate:
    decision = "release16_guarded_reward_short_ppo_mixed_no_promotion"
    next_mechanism = "inspect_failed_folds_then_try_checkpoint_selector_or_feature_repair"
else:
    decision = "release16_guarded_reward_short_ppo_failed_no_promotion"
    next_mechanism = "revise_reward_state_or_target_contract_before_retry"

selection = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": decision,
    "next_mechanism": next_mechanism,
    "canonical_baseline": "soxlbacktest_policy_reimplementation_v1",
    "direct_comparison_baseline": "release16_bull_reentry_participation_objective_replay_v1::bull65_hold3",
    "reward_recipe_baseline": "release16_guarded_reward_checkpoint_objective_v1",
    "profile": PROFILE_NAME,
    "role": "diagnose",
    "changed_layer": ["reward", "bc_to_ppo_handoff", "model_selection"],
    "target_relative_return_pp": TARGET_RELATIVE_RETURN,
    "target_max_dd_pct": TARGET_MAX_DD,
    "strict_context_max_dd_pct": STRICT_MAX_DD,
    "folds_evaluated": evaluated["fold"].tolist() if not evaluated.empty else [],
    "all_folds_evaluated": all_folds_evaluated,
    "selected_gate_pass_count_20dd": selected_gate_count,
    "bc_gate_pass_count_20dd": bc_gate_count,
    "selected_strict_gate_pass_count_15dd": selected_strict_gate_count,
    "bc_strict_gate_pass_count_15dd": bc_strict_gate_count,
    "drift_gate_pass_count": int(evaluated["drift_gate"].fillna(False).astype(bool).sum()) if not evaluated.empty else 0,
    "all_fold_gate": all_fold_gate,
    "all_bc_gate": all_bc_gate,
    "any_drift_fail": any_drift_fail,
    "constrained_selector_decision": constraint_summary.get("decision"),
    "constrained_selector_selected_candidate": constraint_summary.get("selected_candidate"),
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "promotion_blocker": "This is a short 4-iter diagnostic screen. Promotion still requires full fixed-split, complete walk-forward, dashboard/YTD replay, and retention gate review.",
    "reward_patch": reward_patch,
    "sidecars": [
        "coverage.csv",
        "coverage.json",
        "bc_slice_metrics.csv",
        "reward_ppo_selection_history.csv",
        "reward_ppo_fold_results.csv",
        "constrained_checkpoint_metrics.csv",
        "constrained_checkpoint_scores.csv",
        "selection.json",
        "summary.json",
        "conclusion.md",
    ],
}
write_json(CARD / "selection.json", selection)

summary = {
    "experiment": EXPERIMENT,
    "created_at": selection["created_at"],
    "decision": decision,
    "next_mechanism": next_mechanism,
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "selection": selection,
    "coverage": coverage,
    "fold_results": fold_results.to_dict(orient="records"),
    "constrained_selector": {
        "summary": constraint_summary,
        "scores": constraint_scores.to_dict(orient="records") if not constraint_scores.empty else [],
    },
    "source_context": {
        "reward_source_decision": reward_source_summary.get("decision"),
        "prior_bull_decision": prior_bull_summary.get("decision"),
        "prior_bull_selected_candidate": prior_bull_summary.get("selected_candidate"),
        "retention_reference_profile": retention_summary.get("reference_profile"),
        "retention_required_best_return_pct": retention_summary.get("required_best_return_pct"),
    },
    "output_paths": {
        "notebook": rel(CARD / f"{EXPERIMENT}.ipynb"),
        "coverage": rel(CARD / "coverage.json"),
        "fold_results": rel(CARD / "reward_ppo_fold_results.csv"),
        "history": rel(CARD / "reward_ppo_selection_history.csv"),
        "constrained_scores": rel(CARD / "constrained_checkpoint_scores.csv"),
        "official_daily_paths": rel(DAILY_DIR),
        "selection": rel(CARD / "selection.json"),
        "summary": rel(CARD / "summary.json"),
        "conclusion": rel(CARD / "conclusion.md"),
    },
    "conclusion": [],
}

if evaluated.empty:
    summary["conclusion"].append("No folds were evaluated; no training or promotion is allowed.")
else:
    for row in evaluated.to_dict(orient="records"):
        summary["conclusion"].append(
            f"{row['fold']}: BC rel={row.get('bc_test_relative_return', float('nan')):.2f}pp / MaxDD={row.get('bc_test_max_dd', float('nan')):.2f}%; "
            f"reward PPO rel={row.get('selected_test_relative_return', float('nan')):.2f}pp / MaxDD={row.get('selected_test_max_dd', float('nan')):.2f}%; "
            f"delta rel={row.get('ppo_vs_bc_relative_delta', float('nan')):.2f}pp, delta MaxDD={row.get('ppo_vs_bc_max_dd_delta', float('nan')):.2f}pp, "
            f"fold_gate={bool(row.get('fold_gate', False))}."
        )
    summary["conclusion"].append(
        f"Selected reward PPO passes {selected_gate_count}/{len(WALK_FORWARD_WINDOWS)} primary 20% gates and {selected_strict_gate_count}/{len(WALK_FORWARD_WINDOWS)} strict 15% gates; BC initialization passes {bc_gate_count}/{len(WALK_FORWARD_WINDOWS)} primary gates."
    )
    if any_drift_fail:
        summary["conclusion"].append("At least one fold fails the PPO-vs-BC drift guard, so longer PPO from this reward patch is blocked.")
summary["conclusion"].append("No actor is promoted or retained from this notebook.")
write_json(CARD / "summary.json", summary)

conclusion_md = chr(10).join(
    [
        "# Conclusion",
        "",
        f"- decision: `{decision}`",
        f"- selected_gate_pass_count_20dd: `{selected_gate_count}/{len(WALK_FORWARD_WINDOWS)}`",
        f"- bc_gate_pass_count_20dd: `{bc_gate_count}/{len(WALK_FORWARD_WINDOWS)}`",
        f"- promotion_allowed: `{False}`",
        f"- actor_retention_allowed: `{False}`",
        "",
        "## Readout",
        "",
        *[f"- {line}" for line in summary["conclusion"]],
        "",
        "## Next Action",
        "",
        f"- `{next_mechanism}`",
    ]
) + chr(10)
(CARD / "conclusion.md").write_text(conclusion_md, encoding="utf-8")
display(Markdown(conclusion_md))
selection


# Conclusion

- decision: `release16_guarded_reward_short_ppo_drift_failed_no_promotion`
- selected_gate_pass_count_20dd: `1/3`
- bc_gate_pass_count_20dd: `1/3`
- promotion_allowed: `False`
- actor_retention_allowed: `False`

## Readout

- wf_2020: BC rel=22.28pp / MaxDD=21.11%; reward PPO rel=-15.26pp / MaxDD=21.11%; delta rel=-37.54pp, delta MaxDD=0.01pp, fold_gate=False.
- wf_2022: BC rel=72.45pp / MaxDD=15.07%; reward PPO rel=72.45pp / MaxDD=15.07%; delta rel=0.00pp, delta MaxDD=0.00pp, fold_gate=True.
- wf_2023_2024: BC rel=14.30pp / MaxDD=13.96%; reward PPO rel=14.30pp / MaxDD=13.96%; delta rel=0.00pp, delta MaxDD=0.00pp, fold_gate=False.
- Selected reward PPO passes 1/3 primary 20% gates and 0/3 strict 15% gates; BC initialization passes 1/3 primary gates.
- At least one fold fails the PPO-vs-BC drift guard, so longer PPO from this reward patch is blocked.
- No actor is promoted or retained from this notebook.

## Next Action

- `do_not_launch_longer_ppo_from_this_reward_patch; inspect_failed_fold_action_distribution`


{'experiment': 'release16_guarded_reward_short_ppo_screen_v1',
 'created_at': '2026-06-14T21:05:11.365885+00:00',
 'decision': 'release16_guarded_reward_short_ppo_drift_failed_no_promotion',
 'next_mechanism': 'do_not_launch_longer_ppo_from_this_reward_patch; inspect_failed_fold_action_distribution',
 'canonical_baseline': 'soxlbacktest_policy_reimplementation_v1',
 'direct_comparison_baseline': 'release16_bull_reentry_participation_objective_replay_v1::bull65_hold3',
 'reward_recipe_baseline': 'release16_guarded_reward_checkpoint_objective_v1',
 'profile': 'sparse_release_bc_ppo_screen_v1',
 'role': 'diagnose',
 'changed_layer': ['reward', 'bc_to_ppo_handoff', 'model_selection'],
 'target_relative_return_pp': 30.0,
 'target_max_dd_pct': 20.0,
 'strict_context_max_dd_pct': 15.0,
 'folds_evaluated': ['wf_2020', 'wf_2022', 'wf_2023_2024'],
 'all_folds_evaluated': True,
 'selected_gate_pass_count_20dd': 1,
 'bc_gate_pass_count_20dd': 1,
 'selected_strict_gate_pass_count_15dd': 0,
 'bc_str